<!--- Made by:
      Oscar Antonio Restrepo Gutiérrez
--->

# Error theory in computing

It's important to understand that computers do not calculate exactly but, in most cases, approximately. An exact solution is only obtained when working with integers; if the operations involve floating-point numbers, the algorithm's execution will involve approximations due to truncation in the operations (this is a consequence of representing floating-point numbers with 32 or 64 bits). One way to reduce the error is to increase the number of bits (to 128 bits, for example) — that is, more bits means more precision, but never a solution exactly equal to the true one obtained analytically. As we'll see, the algebraic operations of addition and subtraction are not exact ($(a + b) + c \neq a + (b + c)$, i.e. they are not associative) because of the floating-point representation.


## *Overflow* and *underflow*
This refers to the arithmetic overflow that occurs when the binary value stored in a register exceeds its maximum value, i.e. more bits are needed than are available; this happens with very large or very small numbers,
```c
      Overflow }---------------{----0----}--------------{ Overflow
                                underflow
```
The range of a double (64 bits) lies in the interval $10^{-322} < x < 10^{308}$, meaning that
any number outside this range needs more than 64 bits to be represented; if $x$ is very large there is *overflow*, and if it is very small there is *underflow*. The following routine lets us compute *overflow* by multiplying by 2 and *underflow* by dividing by 2,
```python    
i=0.0 
under = over = 1.0
while i<1100:
    under = under/2.0
    over = over*2.0
    print (i, under, over)
    i = i+1
```
If there is *overflow*, python prints *inf*, meaning infinity, and if there is underflow python prints 0.0. Note that once *underflow* is reached, `under` is smaller than the *machine epsilon*.

## Machine epsilon
<a id='epsilon_maquina'></a>
This refers to the smallest (or largest) number that can be added in a sum without changing it, i.e. $x+\epsilon_{min}=x$ (or $x+\epsilon_{max}=\epsilon_{max}$). In other words, *"numbers that differ by less than the machine epsilon, $\epsilon$, are numerically equal"*; this happens because of machine truncation from using floating-point numbers.

In python the machine's numerical precision can be computed with the following routine
```python      
# Machine epsilon: the smallest eps such that 1 + eps != 1
eps = 1 # initial epsilon
for n in range(1200):
    eps = eps/2.0
    one = 1. + eps
    print(n, one, eps)
```
Note that at each step `one` is the sum `1 + eps`, where `eps` is halved at each iteration.
How many iterations are needed for `one` to equal `1.0 + eps`, i.e. for `eps` to become so small that it no longer contributes to the sum?

Python has other additional ways to compute `eps` with greater precision using numpy:
```python 
import numpy as np
eps = np.finfo(float).eps 
1.0 == 1.0 + eps/2. # result: True
```    
(Check these two command lines in ipython to verify that 1 equals 1 + `eps/2`; compare the value of `eps` to the value found in the previous case around iteration 51 — are they the same?).

Likewise, for large numbers, we can find the point at which adding 1.0 stops having any effect:
```python      
# Precision in a sum when the second addend (eps) is much larger
eps=1
for n in range(1200):
    eps=eps*2.0
    one=1.0 + eps
    print(n, one, eps, one-eps) 
```
Note that at each step an `eps` multiplied by 2 is added. As in the previous case, there comes a moment where `one - eps` becomes zero (at what step?), i.e.: `eps` is so large that adding 1.0 doesn't change the result, because that addition falls outside the number's representable range (to see this, run the code and compare `one` and `eps` — at what step does this happen?).


In [1]:
i=0.0 
under = over = 1.0
while i<1100:
    under = under/2.0
    over = over*2.0
    print (i, under, over)
    i = i+1

0.0 0.5 2.0
1.0 0.25 4.0
2.0 0.125 8.0
3.0 0.0625 16.0
4.0 0.03125 32.0
5.0 0.015625 64.0
6.0 0.0078125 128.0
7.0 0.00390625 256.0
8.0 0.001953125 512.0
9.0 0.0009765625 1024.0
10.0 0.00048828125 2048.0
11.0 0.000244140625 4096.0
12.0 0.0001220703125 8192.0
13.0 6.103515625e-05 16384.0
14.0 3.0517578125e-05 32768.0
15.0 1.52587890625e-05 65536.0
16.0 7.62939453125e-06 131072.0
17.0 3.814697265625e-06 262144.0
18.0 1.9073486328125e-06 524288.0
19.0 9.5367431640625e-07 1048576.0
20.0 4.76837158203125e-07 2097152.0
21.0 2.384185791015625e-07 4194304.0
22.0 1.1920928955078125e-07 8388608.0
23.0 5.960464477539063e-08 16777216.0
24.0 2.9802322387695312e-08 33554432.0
25.0 1.4901161193847656e-08 67108864.0
26.0 7.450580596923828e-09 134217728.0
27.0 3.725290298461914e-09 268435456.0
28.0 1.862645149230957e-09 536870912.0
29.0 9.313225746154785e-10 1073741824.0
30.0 4.656612873077393e-10 2147483648.0
31.0 2.3283064365386963e-10 4294967296.0
32.0 1.1641532182693481e-10 8589934592.0
33.0 5.82076

In [2]:
# Machine epsilon: the smallest eps such that 1 + eps != 1
eps = 1 # initial epsilon
for n in range(1200):
    eps = eps/2.0
    one = 1. + eps
    print(n, one, eps)


0 1.5 0.5
1 1.25 0.25
2 1.125 0.125
3 1.0625 0.0625
4 1.03125 0.03125
5 1.015625 0.015625
6 1.0078125 0.0078125
7 1.00390625 0.00390625
8 1.001953125 0.001953125
9 1.0009765625 0.0009765625
10 1.00048828125 0.00048828125
11 1.000244140625 0.000244140625
12 1.0001220703125 0.0001220703125
13 1.00006103515625 6.103515625e-05
14 1.000030517578125 3.0517578125e-05
15 1.0000152587890625 1.52587890625e-05
16 1.0000076293945312 7.62939453125e-06
17 1.0000038146972656 3.814697265625e-06
18 1.0000019073486328 1.9073486328125e-06
19 1.0000009536743164 9.5367431640625e-07
20 1.0000004768371582 4.76837158203125e-07
21 1.000000238418579 2.384185791015625e-07
22 1.0000001192092896 1.1920928955078125e-07
23 1.0000000596046448 5.960464477539063e-08
24 1.0000000298023224 2.9802322387695312e-08
25 1.0000000149011612 1.4901161193847656e-08
26 1.0000000074505806 7.450580596923828e-09
27 1.0000000037252903 3.725290298461914e-09
28 1.0000000018626451 1.862645149230957e-09
29 1.0000000009313226 9.31322574615

In [3]:
# Precisión en la suma cuando el segundo sumando (eps) es mucho más grande
eps = 1 # epsilon inicial
for n in range(1200):
    eps = eps*2.0
    one = 1. + eps
    print(n, one, eps)

0 3.0 2.0
1 5.0 4.0
2 9.0 8.0
3 17.0 16.0
4 33.0 32.0
5 65.0 64.0
6 129.0 128.0
7 257.0 256.0
8 513.0 512.0
9 1025.0 1024.0
10 2049.0 2048.0
11 4097.0 4096.0
12 8193.0 8192.0
13 16385.0 16384.0
14 32769.0 32768.0
15 65537.0 65536.0
16 131073.0 131072.0
17 262145.0 262144.0
18 524289.0 524288.0
19 1048577.0 1048576.0
20 2097153.0 2097152.0
21 4194305.0 4194304.0
22 8388609.0 8388608.0
23 16777217.0 16777216.0
24 33554433.0 33554432.0
25 67108865.0 67108864.0
26 134217729.0 134217728.0
27 268435457.0 268435456.0
28 536870913.0 536870912.0
29 1073741825.0 1073741824.0
30 2147483649.0 2147483648.0
31 4294967297.0 4294967296.0
32 8589934593.0 8589934592.0
33 17179869185.0 17179869184.0
34 34359738369.0 34359738368.0
35 68719476737.0 68719476736.0
36 137438953473.0 137438953472.0
37 274877906945.0 274877906944.0
38 549755813889.0 549755813888.0
39 1099511627777.0 1099511627776.0
40 2199023255553.0 2199023255552.0
41 4398046511105.0 4398046511104.0
42 8796093022209.0 8796093022208.0
43 175921

## Types of errors
<a id='Tipos_de_errores'></a>
First, let's look at the types of errors that occur in computing:

**User errors**: typing mistakes, poor code design, wrong file, etc. — these are avoidable.

**Random errors**: electrical fluctuations, power failures, cosmic rays, etc. These errors are rare, and the probability of them occurring increases with the machine's computation time for the algorithm (some code runs for weeks or months). Unlike the previous case, there is no control over these errors.

**Approximation errors**: these errors are more mathematical in nature, i.e. approximations made when truncating a series or the solution of an equation; the Taylor series is the preferred tool for this type of approximation:

$$\sin(x) = \sum^{N}_{n=1} \frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1} + \epsilon(x,N),$$
where $ \epsilon(x,N)$ is the error introduced by truncating the series.

**Numerical rounding errors**: occurs from only considering a finite number of significant figures in numerical operations. For example, the fractions 2/3 and 1/3 have infinitely many digits in base 10 (and in base 2); truncating to 4 significant figures we have $1/3\approx0.3333$, but $2/3\approx0.6667$, since the last digit is rounded up to 7, so if we do the operation:
$$2\times\frac{1}{3}-\frac{2}{3}=0.6666-0.6667=-\,0.0001\neq0$$
we find an error of 0.0001; although this value is small, it is not zero.

**Rounding errors from using floating-point number types**: This error is basically of the same kind as the previous one; it arises because using a finite 32- or 64-bit representation truncates numbers to about 7 significant figures for 32 bits and about 15 or 16 figures for 64 bits (recall the binary representation). Fortunately, modern computers already use 64-bit representation, which considerably reduces this type of error. Nevertheless, let's look a little more closely at this kind of error and how it affects mathematical approximations:


In [4]:
import numpy as np

# Error in simple operations for 16, 32 and 64 bits:
print( np.float16(5/7.),np.float32(5/7.), 5/7)
print( np.float16(1/10.),np.float32(1/10.), 1/10)


0.7144 0.71428573 0.7142857142857143
0.1 0.1 0.1


In [5]:
# Error in the sum with 16-bit representation,
# compare against the result with 64 bits (used by default)
# note the numpy.floatxx functions round to 16, 32, 64 or 128 bits

# adding 1/10 ten times doesn't give 1.0
x = 0
for i in range(10):
    x += np.float16(1.0/10)
print('operation with 16 bits:',x)


operation with 16 bits: 0.999755859375


**Questions**:<br>
If a number in 16-bit representation uses 5 bits for the exponent and 10 for the mantissa, how many significant figures does it have in base 10?
Which digits are garbage in the calculation above?

**Example**, the following product, $\prod_{n=1}^{20}2^{\frac{1}{20}}$,
should converge to 2; let's compare the results at 16 and 64 bits:


In [6]:
# Error in multiplication with 16-bit representation,
# compare against the result with 64 bits (used by default)
#(subtraction and division are special cases of addition and multiplication.)
# the product converges to 2.0
N = 20
x16 = x64 = 1
print ('iter 16 bits,   64 bits,     error')
for i in range(N):
    x16 *= np.float16(2.0**(1.0/N))
    x64 *= np.float64(2.0**(1.0/N))#Note there's no need to write float64 explicitly
    
print (i, x16, x64, np.abs(x16-x64))


iter 16 bits,   64 bits,     error
19 1.9958053041750938 2.000000000000003 0.004194695824909278


The following example shows the error in computing the series for the sine function,

$$\sin(x) = \sum^{N}_{n=1} \frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1} + \epsilon(x,N),$$

due to using 32 bits instead of 64 bits.
This algorithm is heavy because of the factorial; the error with 32 bits is obvious but not with 64 bits.

Important: note that every number has to be cast to 32 bits for the numerical operation to be done in 32 bits; if any number isn't 32-bit, python will automatically promote all the numbers to 64 bits in the operations, and the error won't be easy to see.


In [7]:
#------ error in computing the sine series with 80 terms----------------
# try other angles!
import numpy as np
import math

x = np.float32(85*np.pi/180) # Initialize   
N = 80                   # a larger value gives an error
f_1 = np.float32(1); f_2 = np.float32(2)
Sum = np.float32(0.0)    # note we use 0.0 (float) and not 0 (integer),
for n in range(1,N+1):   # gives the array 1,2,3, ..... N
   nf = np.float32(n)
   Sum = Sum + (-f_1)**(nf-f_1)*x**(f_2*nf-f_1)/np.float32(math.factorial(2*n-1))

print('sin(x),         series')    
print(np.sin(85*np.pi/180), Sum) # use Sum.dtype to check that it's a float32


sin(x),         series
0.9961946980917455 0.9961946


/var/folders/5y/n_g8hrtd5rn4yxs596rnm8340000gq/T/ipykernel_5013/2095689660.py:12: RuntimeWarning: overflow encountered in cast
  Sum = Sum + (-f_1)**(nf-f_1)*x**(f_2*nf-f_1)/np.float32(math.factorial(2*n-1))


In [8]:
import math
# This algorithm is the same as the previous one, but with 64 bits by default
# and using the numpy.sum() function
x = 85*np.pi/180 # Initialize   
N = 80                     # a larger value gives an error

Sum = sum([(-1.)**(n-1.)*x**(2.*n-1.)/math.factorial(2*n-1) for n in range(1,N+1)])
print (Sum, abs(Sum-np.sin(85*np.pi/180))) 


0.9961946980917457 1.1102230246251565e-16


## How to measure errors
Let $x$ be the true value and $x^*$ the approximate value

**Absolute error**: defined as
\begin{equation*} 
\epsilon_{abs}= |x-x^*|
\end{equation*}
**Relative error**: given by
\begin{equation*} 
\epsilon_{rel}= \frac{|x-x^*|}{|x|}
\end{equation*}
**Percentage error**: given by
\begin{equation*} 
\epsilon_{\%}= \frac{|x-x^*|}{|x|}*100
\end{equation*}
**Error in series**: The error from truncating a series is taken as
\begin{equation*} 
\epsilon_{aprox}= \left|\frac{nth\hbox{-term}}{\hbox{sum}}\right|< \hbox{eps}
\end{equation*}
The tolerance is usually taken to be a small number, e.g. `eps` $=10^{-10}$. Note that the series is not truncated using $|{nth}\hbox{-term}|<$ eps; using this form can lead to errors, since it is not compared against the value of the sum (a million compared to one is large, but compared to ten billion it's small).

Let's again take the sine series calculation as an example and compute the error,


In [9]:
import math
# Only 8 to 9 steps are needed to reach epsilon precision (eps).
x = 85*np.pi/180;  eps = 1.0e-8 # Initialize   
N = 80      # a larger value gives an error
Sum = 0.0   # note we use 0.0 (float) and not 0 (integer),
for n in range(1,N+1): # gives the array 1,2,3, ..... N
   term = (-1)**(n-1)*x**(2.*n-1.)/math.factorial(2*n-1)
   Sum = Sum + term
   if ( abs (term/Sum) < eps ): break # stop if the error is smaller than epsilon
   
print (Sum, n, abs (term/Sum)) 


0.9961946980894644 8 2.848397280372024e-10


Computing a factorial has several important quirks, since it grows very fast (18! already has 16 digits), and although python3 has arbitrary precision for integers, in some cases you have to multiply by a float, and the final result will be truncated to about 16 digits (the rest are lost) — let's see:


In [10]:
import math

# Factorial problem when multiplying by a float:
math.factorial(60), math.factorial(60)*1.0


(8320987112741390144276341183223364380754172606361245952449277696409600000000000000,
 8.32098711274139e+81)

Note that multiplying by 1.0 shifts the decimal point from position 82 to position 1, truncating to 15 digits, which introduces errors into the calculations. Also, the factorial of a not-very-large number produces numeric overflow (for example $171!$ has 310 digits, and $171!\times 1.0$ gives an error, since it's of order $10^{310}$, which is larger than the largest exponent allowed for floats).

### Recycling calculations to reduce error
<a id='reciclaje_de_variable'></a>
Finally, let's highlight the importance of recycling terms calculated in previous iterations. First of all, note that the error in computing the sine series comes from computing, for the *n*th term, the division of two very large quantities, which introduces significant error; in the following variation, computing the factorial is avoided, and the result is better than in the two previous examples. Note that the series term is rewritten as,

$$\frac{(-1)^{n-1}}{(2n-1)!} x^{2n-1}=\frac{-x^2}{(2n-1)(2n-2)} \frac{(-1)^{n-2}}{(2n-3)!} x^{2n-3}$$
(Exercise: verify this identity.)


In [11]:
# Works for x other than zero.
Sum = term = x = 85*np.pi/180; eps = 1.0e-8 # Initialize
n = 2 # we start at 2, since for n = 1 we already assigned the value x
while ( abs (term/Sum) > eps ):
   term = -term*x*x/(2.*n-1.)/(2.*n-2.)
   Sum = Sum + term
   n = n + 1

print('sine function:',np.sin(x))
print('sine series:  ',Sum, n, abs (term/Sum)) 


sine function: 0.9961946980917455
sine series:   0.9961946980894644 9 2.848397280372024e-10


In general the most efficient way to compute the sine series is to recycle
the result from the previous step, avoiding introducing the error caused by an overflow.

**Problem**: The error from not recycling is quite noticeable at 32 bits; at 64 bits it's barely noticeable — repeat this problem using 32 bits and compare.

**Problem**: the sine series code above fails for $x=0$ — how would you fix it?


### Error when adding large and small quantities
When small quantities are added to large ones, the two quantities shouldn't differ by an amount that would require more significant figures than the representation allows; for example, the following sum with 64 bits gives the same result,
```python 
      1.0 + 1e16 = 1e16, 
```      
as explained [earlier](#epsilon_maquina), this happens because we use 64 bits and this representation only allows about 15 significant figures; note that if we add,
```python       
      1.0 + 1.0e15 = 1000000000000001.0
```
the one is added at the last significant digit (in this case the 15th).
This means that addition is not commutative for floating-point numbers, i.e. $(a + b) + c \neq a + (b + c)$.      


In [12]:
# The result depends on how the added ones are associated:
print(1e16 + 1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1  )
print(1e16 +(1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1+1) )
print(1e16 +(1+1+1+1+1+1+1+1+1+1+1+1+1)+1+1+1+1+1+1+1+1+1+1+1 )


1e+16
1.0000000000000024e+16
1.0000000000000012e+16


### Subtractive cancellation error
<a id='Error_de_cancelación_sustractiva'></a>
Occurs when subtracting two very similar numbers, because the result ends up with fewer significant figures; for example, consider the following operation with 64 bits,
```python 
       1234567891 - 1234567809 = 82
```      
but with 32 bits it gives,
```python 
      float32(1234567891) - float32(1234567809) = 128.0
```
If this subtraction is in the denominator of an operation, the error is even worse (compute the reciprocals and compare).

The precision of the cancellation result will equal the number of nonzero digits on the right; for example, the subtraction $1234999-1234888=0000111$ has only 3 significant figures.

**Example**


In [13]:
# Operation with 64 bits
x = 1234567891.0 
y = 1234567809.0 
print(x - y)   # gives 82.0

# Operation with 32 bits
x32 = np.float32(x)
y32 = np.float32(y)
print(x32 - y32) # gives 128.0 


82.0
128.0


**Exercise**: compare the following two quantities, where the first 8 digits are equal; subtract them to see that the precision of the result is eight digits (the last 8 digits) and not 16:


In [14]:
# Compare the following quantities for int and float type
# first 8 digits equal, the subtraction gives: 00000000806398288.0
12345678901234567, 12345678094836279,  12345678901234567., 12345678094836279. 


(12345678901234567,
 12345678094836279,
 1.2345678901234568e+16,
 1.234567809483628e+16)

**Example**: comparison of two quantities with 17 equal digits:


In [15]:
# first 17 digits equal with integers: true result
123456789012345675454325 - \
123456789012345678794305 # 


-3339980

In [16]:
# first 17 digits equal, but with floats: wrong result
123456789012345675454325. - \
123456789012345678794305.


-16777216.0

**Catastrophic error in the quadratic equation**: this is a classic example of subtractive cancellation; consider the quadratic equation

$$ax^{2}+bx+c=0$$

which has the exact solution

$$x=\frac{-b \pm \sqrt{b^2-4ac} }{2a}.$$

Subtractive cancellation error appears in one of the roots when $b^2>>4ac$, because inside the square root its terms roughly cancel out (which gives $b \pm\sqrt{b^2}\approx 0$ depending on the case). One solution is to rewrite the solution as

$$x=\frac{-2c}{b \pm \sqrt{b^2-4ac} }.$$

This expression has greater precision, since for example if $b>0$, then $b+\sqrt{b^2}\approx 2b$; likewise in the other case, $b<0$, since $-b-\sqrt{b^2}\approx -2b$ (see exercise at the end).

**Example**: Consider the following equation,

$$x^2 - 1.786737601482363x + 2.054360090947453\times 10^{-8}=0$$

which has analytical roots (double representation with 16 digits of precision),

$$(x-1.786737589984535)(x-1.149782767465722\times 10^{-8})=0$$

Let's compare the two solution methods:


In [17]:
## solving the equation ax^2 + bx + c = 0
# note that b**2 >> 4ac
x1=1.786737589984535    # analytical root
x2=1.149782767465722e-8 # 
a=1.; b = - 1.786737601482363; c = 2.054360090947453e-8 

print( b**2,'>>', 4*a*c)

K = b**2. - 4.*a*c
x_plus  = (-b + K**0.5)/(2*a)# note that b<0, so no cancellation here
x_minus = (-b - K**0.5)/(2*a)# root with cancellation problems

y_plus  = -2.*c/(b + K**0.5) # doesn't work, since it introduces cancellation (would work if b>0).
y_minus = -2.*c/(b - K**0.5) # fixes the cancellation problem
print ("Normal formula:    ",x_minus, x_plus)
print ("Modified formula:  ",y_minus, y_plus)
print ("true value:        ",x2, x1)


3.1924312565509476 >> 8.217440363789811e-08
Normal formula:     1.1497827689943563e-08 1.7867375899845355
Modified formula:   1.1497827674657215e-08 1.78673758760907
true value:         1.149782767465722e-08 1.786737589984535


Looking closely, we see that for `x_minus` the normal formula only retains 8 significant figures due to subtractive cancellation between $b$ and $\sqrt{(b^2-4ac)}$, but using the modified formula recovers all the digits. So the solution is given by the value of `x_plus` from the original formula and `y_minus` from the modified formula.

Note that the formula above only avoids cancellation between $b$ and $\sqrt{(b^2-4ac)}$, but not cancellation inside the root $b^2-4ac$ itself; in these cases you'd need to double the precision (128 bits) if a very precise result is required. Let's see an example — consider the equation proposed by Kahan,

$$94906265.625x^2-189812534x+94906268.375=0$$

with roots

 $$(x - 1.000000028975958)(x- 1.000000000000000)=0$$ 

If we solve it in python we see that both formulas give wrong results:


In [18]:
a=94906265.625; b=-189812534; c = 94906268.375

K = b**2. - 4*a*c
x_plus  = (-b + K**0.5)/(2*a) 
x_minus = (-b - K**0.5)/(2*a)  

y_plus  = -2*c/(b + K**0.5) 
y_minus = -2*c/(b - K**0.5)  

print ("Normal formula:    ",x_plus, x_minus)
print ("Modified formula:  ",y_plus, y_minus)


Normal formula:     1.0000000144879793 1.0000000144879793
Modified formula:   1.000000014487979 1.000000014487979


# Supplement
Definition of [ULP](https://en.wikipedia.org/wiki/Unit_in_the_last_place) ("*Unit in the Last Place*" or "*Unit of Least Precision*"): a measure of the spacing between floating-point numbers, used to measure precision in numerical calculations.

<!---
In computer science and numerical analysis, unit in the last place or unit of least precision (ULP) is the spacing between floating-point numbers, i.e., the value the least significant digit (rightmost digit) represents if it is 1. It is used as a measure of accuracy in numeric calculations.
also see:
https://math.stackexchange.com/questions/42920/efficient-and-accurate-approximation-of-error-function
--->
In the following example the result is $2^{53}$, due to the double-precision format that uses 53 bits for the significand:


In [19]:
x = 1.0 # initial value
n = 0   # exponent of 2^n
while x != x + 1: 
    x = x * 2 
    n = n + 1 

x, n 


(9007199254740992.0, 53)


## Exercises
1) Compute the absolute and relative error of $p$ and $p^∗$:

&emsp; a) $p = \pi,\, p^∗ = 22/7$<br> 
&emsp; b) $p = \pi,\, p^∗ = 3.1416$<br>
&emsp; c) $p = e,\, p∗ = 2.718$<br> 
&emsp; d) $p = \sqrt{2},\, p∗ = 1.414$<br>
&emsp; e) $p = e^{10},\, p^∗ = 22000$<br> 
&emsp; f) $p = 10^{\pi} ,\, p^∗ = 1400$<br>
&emsp; g) $p = 8!\,, p^∗ = 39900$<br> 
&emsp; h) $p = 9!,\, p^∗ = \sqrt{18\pi}(9/e)^9$

2) Consider the following code:
```python      
for x in range(20):
    print (x,10**x + 1.0e20)
```
When run, the first 4 prints are equal — explain why.
What happens if 32 bits are used in the operation?

3) What is the smallest quantity, $a$, that can be added to the sum so that it changes (i.e. $x+a\neq x$) for the following numbers?
```python 
1.0e10
1.0e15
1.0e20
1.025e30
```        
if a) the numbers are 64-bit, b) they are 32-bit?

4) Explain why $ (1000 + 0.5) + 0.5 = 1000 + 0.5$ gives a different answer than $1000. + (0.5 + 0.5)$ if only 4 significant figures are considered — what are the results in both cases?

5) investigate how subtractive cancellation affects the following operation
$$\frac{f(b)-f(a)}{b-a},$$ if $f(x)=x^2, b=3.0$ and

&emsp; a) $b-a=0.001$,<br> 
&emsp; b) $b-a=0.00001$,<br> 
&emsp; c) $b-a=0.000001$.  

6) Implement the following expressions and series in python in the traditional way, then use the [recycling](#reciclaje_de_variable) idea from the previous step (as was done for the sine series) to reduce the error and simplify the calculations (use `eps = 1e-8`, $x$ = 45 and $N=100$); compute the error in each case,

&emsp; a) $e^x = \sum_{n=0}^N \frac{x^n}{n!}$. 

&emsp; b) $\binom {n}{k}={\frac {n!}{k!\,(n-k)!}}={\frac {n+1-k}{k}}{\binom {n}{k-1}}$, recurrence formula for the binomial coefficient (consider the symmetry: $\binom {n}{k} =\binom {n}{n-k}$).

&emsp; c) $B_{k}=-\sum _{i=0}^{k-1}{k \choose {i}}{\frac {B_{i}}{k+1-i}}$, with $B_0=1$, recurrence formula for Bernoulli numbers.

&emsp; d) $\cos x = \sum^{N}_{n=0} \frac{(-1)^n}{(2n)!} x^{2n}$.
     
&emsp; e) $\tan x = \sum^{N}_{n=1} \frac{B_{2n} (-4)^n \left(1-4^n\right)}{(2n)!} x^{2n-1}$, where the $B_k$ are the Bernoulli numbers.
     
&emsp; f) $\text{arcsin}\, x = \sum^{N}_{n=0} \frac{(2n)!}{4^n (n!)^2 (2n+1)} x^{2n+1},\quad\mbox{ for } \left| x \right| < 1$. 
     
&emsp; g) $\arccos x =\frac{\pi}{2}-\text{arcsin}\, x =\frac{\pi}{2}- \sum^{N}_{n=0} \frac{(2n)!}{4^n (n!)^2 (2n+1)} x^{2n+1}$.

&emsp; h) $\arctan x = \sum^N_{n=0} \frac{(-1)^n}{2n+1} x^{2n+1}\quad\mbox{, for } \left| x \right| < 1.$

&emsp; i) $(x+y)^{n}=\sum _{k=0}^{n}{\binom {n}{k}}x^{n-k}y^{k}$, binomial formula.

&emsp; j) $\sum _{k=0}^{n}{\binom {n}{k}}=2^{n}$.

7) Pascal's triangle is determined from the coefficients of the expansion of the binomial formula
given in the previous exercise, i.e.:
$$ 
\begin{eqnarray}
(a+b)^{0}&=&\quad\quad\quad\quad\,\,\, 1\\
(a+b)^{1}&=&\quad\quad\quad\,\, 1a+1b\\
(a+b)^{2}&=&\quad\quad\, 1a^{2}+2ab+1b^{2}\\
(a+b)^{3}&=&\quad 1a^{3}+3a^{2}b+3ab^{2}+1b^{3}\\
&\vdots&\\
(a+b)^{n}&=&1a^{n}+a^{n-1}b+a^{n-2}b^{2}...+ 1b^{n},\\
\end{eqnarray}
$$

write a program that plots Pascal's triangle for $n=10$.

8) Write a program that computes the 2 roots of the quadratic equation using the traditional numerical solution and another using the higher-precision solution, and compare the results. a) For this use the equation $8.47x^2+52.31x+0.3904=0.0$.

b) Investigate how the error changes as subtractive cancellation approaches the machine epsilon (use $a=1,b=1$, $c=10^{-n}$ with $n=1,2,3,...$).

9) [Kahan summation](https://en.wikipedia.org/wiki/Kahan_summation_algorithm) is a numerical method that significantly reduces the error when adding small quantities to large ones in an array or list of floating-point numbers. a) Implement Kahan summation given by the pseudocode,
```c
function KahanSum(input)  // input is an array of dimension N.
    sum = 0.0             // Prepare the accumulator.
    c = 0.0               // A running compensation for lost low-order bits.
    for i = 1 to N do     // The array input has elements indexed input[1] to input[N].
        y = input[i] - c  // c is zero the first time around.
        t = sum + y       // sum is big, y small, so low-order digits of y are lost.
        c = (t - sum) - y // (t - sum) cancels the high-order part of y; subtracting y recovers negative (low part of y)
        sum = t           // Algebraically, c should always be zero. Beware overly-aggressive optimizing compilers!
    next i                // Next time around, the lost low part will be added to y in a fresh attempt.
    return sum
```  
b) Verify that if `eps=1.1102230246251565e-16`, 
```c
(1.0 + eps) - eps gives 0.9999999999999999
```
but Kahan summation gives 1.0 (find your computer's `eps` and test it).<br>
c) Show that if $a, b, c$ are $10000.0, 3.14159, 2.71828$, then $(a + b) + c$ gives the value $10005.8$, but Kahan summation gives the more precise value $10005.9$. (Note: to see this difference you'll need to work with only 6 significant figures in python3; to do this use,
```python
>>> from decimal import *
>>> getcontext().prec = 6
>>> a, b, c = [Decimal(n) for n in '10000.0 3.14159 2.71828'.split()]
```
10) When the following array is run in python,
```python
np.array([1., 10**100, 1., -10**100]).sum() gives 0.0
```
but a simple inspection of the array shows the correct result is 2.0. Show that Kahan summation fails but the *Neumaier* algorithm gives the correct result:

```c
function NeumaierSum(input)           // input is an array of dimension N.
    sum = 0.0
    c = 0.0                           // A running compensation for lost low-order bits.
    for i = 1 to N do
        t = sum + input[i]
        if |sum| >= |input[i]| then
            c += (sum - t) + input[i] // If sum is bigger, low-order digits of input[i] are lost.
        else
            c += (input[i] - t) + sum // Else low-order digits of sum are lost
        endif
        sum = t
    next i
    return sum + c                    // Correction only applied once in the very end
```

11) Archimedes' formula approximates the number $\pi$ by computing perimeters inscribed in a circle as,

$$
\pi \sim 6t_{i}2^{i} 
$$, 

with, $i=0,1, ...,n,$ where,

$$
t_{i+1}=\frac{{\sqrt {t_{i}^{2}+1}}-1}{t_{i}}.
$$

a) Show that $t_{i+1}$ can be rewritten as,

$$
t_{i+1}=\frac{t_{i}}{{\sqrt {t_{i}^{2}+1}}+1}.
$$

b) For $n=30$ and $t_{0}={\frac{1}{\sqrt{3}}}$ compare the two formulas — which is more accurate? Plot the relative error and explain why one is more accurate than the other (use as the theoretical value $\pi= 3.14159265358979323846$).



**Bibliography**:

Landau, Páez, *"A Survey of
Computational Physics
Introductory Computational Science"*, chap 1, 2.

Burden, *Numerical Analysis*, chap 1.

https://ece.uwaterloo.ca/~dwharder/NumericalAnalysis/02Numerics/Weaknesses/

https://en.wikipedia.org/wiki/Loss_of_significance

https://en.wikipedia.org/wiki/Floating-point_arithmetic#Floating-point_arithmetic_operations
